In [8]:
import os
from dotenv import load_dotenv
from langchain_groq import  ChatGroq
from langchain_core.documents import  Document
from langchain_text_splitters import  RecursiveCharacterTextSplitter
from langchain_huggingface import  HuggingFaceEmbeddings
from langchain_astradb import AstraDBVectorStoreError

In [9]:


load_dotenv()
# Set your Astra DB credentials
os.environ["ASTRA_DB_API_ENDPOINT"] = os.getenv("Astra_DB_API_Endpoint")
os.environ["ASTRA_DB_APPLICATION_TOKEN"] = os.getenv("Astra_DB_Token")
load_dotenv()

os.environ["LANGCHAIN_TRACING_V2"] = "true"
os.environ["LANGCHAIN_API_KEY"] = os.getenv("LANGSMITH_API_KEY")
os.environ["LANGCHAIN_PROJECT"] = os.getenv("LANGSMITH_PROJECT")

os.environ["GROQ_API_KEY"] = os.getenv("GROQ_API_KEY")

llm = ChatGroq(model="openai/gpt-oss-120b")


In [2]:
from langchain_astradb  import AstraDBVectorStore
from langchain_huggingface import  HuggingFaceEmbeddings


emb=HuggingFaceEmbeddings(model_name="BAAI/bge-small-en-v1.5")
vectore_store=AstraDBVectorStore(

embedding=emb,
collection_name="agent_work",

api_endpoint= os.getenv("Astra_DB_API_Endpoint"),
token=os.getenv("Astra_DB_Token")
)

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

In [10]:
from langchain_core.documents import  Document
from langchain_text_splitters import  RecursiveCharacterTextSplitter

sample_texts = [
    "Astra DB is a serverless vector database built on Apache Cassandra.",
    "LangGraph allows you to build stateful, multi-actor applications with LLMs.",
    "RAG systems combine retrieval of private data with generative AI models to prevent hallucinations."
]
docs=[Document(page_content=text,metadata={"source":"notebook_test"}) for text in sample_texts]

text_spliter=RecursiveCharacterTextSplitter(chunk_size=500,chunk_overlap=50)
splits_docs=text_spliter.split_documents(docs)

splits_docs


[Document(metadata={'source': 'notebook_test'}, page_content='Astra DB is a serverless vector database built on Apache Cassandra.'),
 Document(metadata={'source': 'notebook_test'}, page_content='LangGraph allows you to build stateful, multi-actor applications with LLMs.'),
 Document(metadata={'source': 'notebook_test'}, page_content='RAG systems combine retrieval of private data with generative AI models to prevent hallucinations.')]

In [11]:
insert_ids=vectore_store.add_documents(splits_docs)

In [13]:
print(f"Inserted {len(insert_ids)} document chunks into Astra DB.")

Inserted 3 document chunks into Astra DB.


In [15]:
query = "What database is serverless and uses Cassandra?"


res=vectore_store.similarity_search_with_score(query,k=1)
for doc ,score in res:
    print(f"score :{score} the doc:{doc}")


    


score :0.92369384 the doc:page_content='Astra DB is a serverless vector database built on Apache Cassandra.' metadata={'source': 'notebook_test'}


In [ ]:
retriver=vectore_store.as_retriever(search_kwargs={"k":2})
ress=retriver.invoke("tell me about the RAG ")


for  i , doc in enumerate(ress):
    print(f"res:{i+1}:{ress.page_content}")



    
